In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

In [3]:
data_df = pd.read_csv("../processed_data/data_combined_training.csv", index_col=0)
data_df

,App A,App B,Num Nodes,App A Isolated Time,App B Isolated Time,App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.2_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.2_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.4_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.4_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.6_Mode=d],...,App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.4_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.4_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.6_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.6_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.8_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.8_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=1.0_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=1.0_Mode=d],App A Co-Scheduled Time with App B,App B Co-Scheduled Time with App A
0,beatnik,fiesta,1,577.802632,329.061404,758.671053,581.881579,748.350877,580.969298,748.785088,...,747.324561,581.846491,750.820175,581.359649,758.592105,581.960526,748.438596,582.320175,739.087719,582.535088
1,beatnik,lammps,1,577.802632,422.631579,758.671053,528.070175,748.350877,523.912281,748.785088,...,747.324561,524.350877,750.820175,515.543860,758.592105,522.982456,748.438596,530.859649,738.333333,515.105263
2,beatnik,minife,1,577.802632,408.052632,758.671053,749.122807,748.350877,752.157895,748.785088,...,747.324561,749.122807,750.820175,748.877193,758.592105,747.197368,748.438596,748.438596,737.855263,748.877193
3,beatnik,minivite,1,577.802632,560.508772,758.671053,618.877193,748.350877,620.789474,748.785088,...,747.324561,614.964912,750.820175,619.368421,758.592105,634.438596,748.438596,617.508772,745.978070,614.035088
4,beatnik,tricount,1,577.802632,466.421053,758.671053,551.263158,748.350877,549.403509,748.785088,...,747.324561,572.789474,750.820175,549.403509,758.592105,554.245614,748.438596,551.456140,734.293860,551.263158
5,fiesta,lammps,1,329.061404,422.631579,581.881579,528.070175,580.969298,523.912281,586.065789,...,581.846491,524.350877,581.359649,515.543860,581.960526,522.982456,582.320175,530.859649,580.938596,517.403509
6,fiesta,minife,1,329.061404,408.052632,581.881579,749.122807,580.969298,752.157895,586.065789,...,581.846491,749.122807,581.359649,748.877193,581.960526,747.197368,582.320175,748.438596,585.447368,749.859649
7,fiesta,minivite,1,329.061404,560.508772,581.881579,618.877193,580.969298,620.789474,586.065789,...,581.846491,614.964912,581.359649,619.368421,581.960526,634.438596,582.320175,617.508772,581.921053,623.771930
8,fiesta,tricount,1,329.061404,466.421053,581.881579,551.263158,580.969298,549.403509,586.065789,...,581.846491,572.789474,581.359649,549.403509,581.960526,554.245614,582.320175,551.456140,581.535088,555.421053
9,lammps,minife,1,422.631579,408.052632,528.070175,749.122807,523.912281,752.157895,524.105263,...,524.350877,749.122807,515.543860,748.877193,522.982456,747.197368,530.859649,748.438596,522.736842,747.701754


In [5]:
cols_to_scale = data_df.columns[2:]
scaler = StandardScaler()
data_scaled = data_df.copy()
data_scaled[cols_to_scale] = scaler.fit_transform(data_df[cols_to_scale])
data_scaled

,App A,App B,Num Nodes,App A Isolated Time,App B Isolated Time,App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.2_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.2_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.4_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.4_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=0us_Sparsity=0.6_Mode=d],...,App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.4_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.4_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.6_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.6_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.8_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=0.8_Mode=d],App A Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=1.0_Mode=d],App B Co-Scheduled Time with Inhib [MsgSz=35000000_Wait=1000000us_Sparsity=1.0_Mode=d],App A Co-Scheduled Time with App B,App B Co-Scheduled Time with App A
0,beatnik,fiesta,0.0,1.182366,-2.012262,1.065623,-0.335210,1.021027,-0.335774,1.029124,...,1.028825,-0.431720,1.032145,-0.304329,1.059069,-0.385923,1.038080,-0.332229,0.996211,-0.308948
1,beatnik,lammps,0.0,1.182366,-0.625551,1.065623,-1.030783,1.021027,-1.052590,1.029124,...,1.028825,-1.217182,1.032145,-1.131600,1.059069,-1.146156,1.038080,-1.003796,0.988016,-1.160091
2,beatnik,minife,0.0,1.182366,-0.841612,1.065623,1.826572,1.021027,1.814894,1.029124,...,1.028825,1.853485,1.032145,1.801282,1.059069,1.743997,1.038080,1.835639,0.982822,1.790729
3,beatnik,minivite,0.0,1.182366,1.417789,1.065623,0.143000,1.021027,0.164493,1.029124,...,1.028825,0.020720,1.032145,0.173422,1.059069,0.290525,1.038080,0.126987,1.071064,0.088665
4,beatnik,tricount,0.0,1.182366,0.023409,1.065623,-0.730988,1.021027,-0.732340,1.029124,...,1.028825,-0.555450,1.032145,-0.706001,1.059069,-0.743171,1.038080,-0.735009,0.944133,-0.703683
5,fiesta,lammps,0.0,-1.245200,-0.625551,-0.748327,-1.030783,-0.731458,-1.052590,-0.695307,...,-0.720781,-1.217182,-0.695045,-1.131600,-0.739289,-1.146156,-0.748116,-1.003796,-0.721841,-1.131081
6,fiesta,minife,0.0,-1.245200,-0.841612,-0.748327,1.826572,-0.731458,1.814894,-0.695307,...,-0.720781,1.853485,-0.695045,1.801282,-0.739289,1.743997,-0.748116,1.835639,-0.672860,1.803130
7,fiesta,minivite,0.0,-1.245200,1.417789,-0.748327,0.143000,-0.731458,0.164493,-0.695307,...,-0.720781,0.020720,-0.695045,0.173422,-0.739289,0.290525,-0.748116,0.126987,-0.711168,0.211570
8,fiesta,tricount,0.0,-1.245200,0.023409,-0.748327,-0.730988,-0.731458,-0.732340,-0.695307,...,-0.720781,-0.555450,-0.695045,-0.706001,-0.739289,-0.743171,-0.748116,-0.735009,-0.715361,-0.651199
9,lammps,minife,0.0,-0.332011,-0.841612,-1.300460,1.826572,-1.328846,1.814894,-1.351938,...,-1.328685,1.853485,-1.365859,1.801282,-1.339769,1.743997,-1.301448,1.835639,-1.354115,1.775892


In [7]:
# Principal component analysis
pca = PCA(n_components=3)
P_pca = pca.fit_transform(data_scaled[data_scaled.columns[2:]])

In [8]:
P_pca

array([[  5.89967249,   2.98455888,  -1.98349572],
       [  9.02573132,  -0.32158753,  -0.79780086],
       [ -3.016061  ,  12.25244243,  -0.59165748],
       [  3.97516671,   4.91038901,   1.33948412],
       [  7.46277167,   1.27564876,  -0.14394217],
       [  1.27829867,  -7.73385661,  -0.5896828 ],
       [-10.75510504,   4.84437119,  -0.38261675],
       [ -3.79172587,  -2.49885383,   1.54884528],
       [ -0.28134139,  -6.12803206,   0.06637721],
       [-13.32299441,   2.3870501 ,  -0.87102673],
       [ -6.35390692,  -4.95653293,   1.06021106],
       [ -2.8388611 ,  -8.58666258,  -0.42261482],
       [  3.81418124,   4.79768292,   1.88792138],
       [  7.34645082,   1.15017248,   0.40009801],
       [  1.55772282,  -4.37679023,  -0.52009973]])